# 🎬 SlideForge AI — Notebook Playground

Drive the **agentic slide pipeline** programmatically — everything the Streamlit app does, from Python.

```
Brief ──▶ Researcher ─▶ Planner ─▶ Writer (per slide) ─▶ Critic ⇄ Reviser ─▶ .pptx
                │                                            │
                └────────── Designer (auto theme pick) ──────┘
```

**Bring your own key** — set `OPENAI_API_KEY` in the environment / a `.env` file, or paste it when prompted below. Works with any OpenAI-compatible endpoint (Groq, OpenRouter, local vLLM…) via `base_url`.

In [ ]:
# If running for the first time:  %pip install -r ../requirements.txt
import sys
from pathlib import Path

# Make the repo importable whether the notebook runs from notebooks/ or the repo root
ROOT = Path.cwd() if (Path.cwd() / "slide_agent").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from slide_agent import (
    AgentConfig, LLMClient, SlideAgent, THEMES, Theme,
    build_deck, create_presentation, get_theme,
)
print("SlideForge loaded from:", ROOT)

In [ ]:
import os
from getpass import getpass

try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    pass

API_KEY = os.environ.get("OPENAI_API_KEY") or getpass("Paste your OpenAI API key: ")
MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
BASE_URL = os.environ.get("OPENAI_BASE_URL") or None  # e.g. "https://api.groq.com/openai/v1"

llm = LLMClient(api_key=API_KEY, model=MODEL, base_url=BASE_URL, temperature=0.7)
print("Endpoint check:", llm.ping())

## 1 · Plan an outline
The **Researcher** builds a fact sheet, then the **Planner** shapes the narrative arc and picks a layout type per slide.

In [ ]:
BRIEF = (
    "A 10-slide deck for bank executives on adopting agentic AI in corporate treasury: "
    "the opportunity, top use cases (cash forecasting, hedging support, payment anomaly detection), "
    "reference architecture, risks & governance, and a 12-month adoption roadmap."
)

config = AgentConfig(
    n_slides=10,
    audience="Bank executives (CFO, Head of Treasury)",
    tone="Executive & crisp",
    language="English",
    research=True,
    critique_rounds=1,
)
agent = SlideAgent(llm, config)

notes = agent.research(BRIEF)
outline = agent.plan(BRIEF, notes)

print(f"📋 {outline.deck_title} — {outline.subtitle}\n")
for i, s in enumerate(outline.slides, 1):
    print(f"{i:>2}. [{s.type:<10}] {s.title}")

## 2 · Edit the outline (optional)
The outline is plain data — tweak titles, reorder slides, or change a slide's `type` before writing.

In [ ]:
# Example edit: make sure slide 2 is a KPI slide about the market opportunity
if len(outline.slides) > 1:
    outline.slides[1].type = "kpi"
    outline.slides[1].hints += " Lead with 3-4 headline market figures."
outline.slides[1] if len(outline.slides) > 1 else outline

## 3 · Write → Critique → Revise → Build
Watch each agent hand off to the next via the event callback.

In [ ]:
def on_event(stage, msg, frac):
    print(f"[{frac:>4.0%}] {stage:<9} {msg}")

result = agent.run_from_outline(BRIEF, outline, on_event)
deck = result.deck
print(f"\n✅ {len(deck.slides)} slides · {deck.word_count} words")
print("Critique log:", *result.critique_log, sep="\n  ")

In [ ]:
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)

pptx_bytes = build_deck(deck, theme=get_theme("boardroom"), footer="SlideForge AI · Confidential")
path = OUT / "treasury_agentic_ai.pptx"
path.write_bytes(pptx_bytes)
print(f"💾 Saved {path} ({len(pptx_bytes)/1024:.0f} KB)")

## 4 · The theme gallery
Render the *same* deck in every built-in design system — or define your own `Theme` in one line.

In [ ]:
for key, theme in THEMES.items():
    print(f"• {theme.name:<12} {theme.tagline}")

# Render the whole gallery (one file per theme):
for key in THEMES:
    (OUT / f"gallery_{key}.pptx").write_bytes(build_deck(deck, theme=THEMES[key]))
print("\n💾 Gallery written to", OUT)

In [ ]:
# Your own brand system — just colours + fonts:
my_brand = Theme(
    key="acme", name="Acme Corp", tagline="Acme brand system",
    bg="FFFFFF", surface="F0F7F4", text="0B3D2E", muted="5E7D72",
    primary="0B3D2E", secondary="14795B", accent="FF7A00",
    heading_font="Segoe UI", body_font="Calibri",
)
(OUT / "acme_brand.pptx").write_bytes(build_deck(deck, theme=my_brand))
print("💾 acme_brand.pptx written")

## 5 · Bring your own template
Drop a corporate `.pptx`/`.potx` into `templates/` — SlideForge keeps its masters, layouts, colours and fonts, clears old slides, and writes the new deck *into* it.

In [ ]:
from slide_agent.template_analyzer import describe_template, open_template

candidates = sorted((ROOT / "templates").glob("*.pptx")) + sorted((ROOT / "templates").glob("*.potx"))
if candidates:
    tpl = candidates[0]
    data = tpl.read_bytes()
    info = describe_template(open_template(data, tpl.name))
    print(f"🖼️ {tpl.name}: {len(info['layouts'])} layouts, colours {info['colors']}")
    (OUT / f"branded_{tpl.stem}.pptx").write_bytes(
        build_deck(deck, template_bytes=data, template_name=tpl.name)
    )
    print("💾 Branded deck written to output/")
else:
    print("No template found — drop a .pptx/.potx into templates/ and re-run this cell.")

## 6 · One-liner
The whole pipeline — research → plan → write → critique → build — in a single call.

In [ ]:
pptx_bytes, res = create_presentation(
    "A 7-slide explainer on how RAG works, for non-technical product managers",
    api_key=API_KEY, model=MODEL, base_url=BASE_URL,
    n_slides=7, audience="Product managers", tone="Educational",
    auto_theme=True,          # let the Designer agent choose the look
    critique_rounds=1,
)
(OUT / "rag_explainer.pptx").write_bytes(pptx_bytes)
print(f"💾 rag_explainer.pptx · Designer chose theme: {res.theme_choice}")
print(f"Tokens used: {llm.usage}")

---
### Where to go next
- `streamlit run app.py` — the full visual studio (editable outline grid, theme gallery, template upload)
- Add a slide archetype: extend `SlideType` in `slide_agent/models.py`, teach the Writer in `agent.py`, paint it in `builder.py`
- Point `BASE_URL` at Groq/OpenRouter/vLLM to try open-weights models — no code changes needed